# FinTok AI · 04_video / 02 · Avatar realista con EchoMimic en local (Colab)

**Objetivo:** generar el vídeo de la cara hablando con **EchoMimic** (versión acelerada) en la GPU gratuita de Colab, sin depender de la cuota de los Spaces de Hugging Face.

**Problema técnico:** EchoMimic está hecho para Python 3.8–3.10 y versiones antiguas de PyTorch y diffusers, y Colab usa Python 3.13.
**Solución:** creamos con `uv` un entorno aparte con Python 3.10 y ejecutamos EchoMimic dentro de él, sin tocar las librerías del notebook.

**Entrada:** el avatar (`andrea_closed.png`) y el JSON de escenas.
**Salida:** un vídeo por escena en `outputs/avatar_videos/` y el JSON con el campo `avatar_video_path`, listo para el montador.

⚠️ Necesita **GPU T4** (Entorno de ejecución → Cambiar tipo de entorno).

## 0. Comprobar la GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Entorno Python 3.10 para EchoMimic (~5 min)

Versiones fijadas y comprobadas para que sean compatibles entre sí.

In [ ]:
%%writefile /content/echomimic_requirements.txt
torch==2.2.2
torchvision==0.17.2
torchaudio==2.2.2
numpy<2
diffusers==0.24.0
huggingface_hub==0.25.2
transformers==4.38.2
accelerate==0.27.2
omegaconf==2.3.0
einops==0.4.1
av>=12,<14
moviepy==1.0.3
ffmpeg-python==0.2.0
opencv-python-headless<4.11
facenet_pytorch==2.5.0
more-itertools
regex
tqdm
safetensors

In [ ]:
!pip install -q uv
!uv venv -q --python 3.10 /content/emenv
!VIRTUAL_ENV=/content/emenv uv pip install -q -r /content/echomimic_requirements.txt
!/content/emenv/bin/python -c "import torch, diffusers; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| diffusers', diffusers.__version__)"

## 2. Código y pesos del modelo (~10 GB, unos minutos)

Solo descargamos los pesos de la versión acelerada de audio→vídeo (el repositorio completo pesa más de 60 GB).

In [ ]:
!git clone -q https://github.com/BadToBest/EchoMimic /content/EchoMimic

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    "BadToBest/EchoMimic",
    local_dir="/content/EchoMimic/pretrained_weights",
    allow_patterns=[
        "sd-image-variations-diffusers/unet/*",
        "sd-vae-ft-mse/*",
        "audio_processor/whisper_tiny.pt",
        "denoising_unet_acc.pth",
        "reference_unet.pth",
        "face_locator.pth",
        "motion_module_acc.pth",
    ],
)
!du -sh /content/EchoMimic/pretrained_weights

## 3. Subir el avatar y el guion

Sube `andrea_closed.png` y el JSON de escenas (por ejemplo `aapl_2024.json`).

In [ ]:
import os, json, glob, shutil
from google.colab import files
from PIL import Image

uploaded = files.upload()
JSON_PATH = [f for f in uploaded if f.endswith(".json")][0]
AVATAR_PNG = [f for f in uploaded if f.endswith(".png")][0]
spec = json.load(open(JSON_PATH, encoding="utf-8"))
print(spec["company"], spec["year"], "·", len(spec["scenes"]), "escenas · avatar:", AVATAR_PNG)

## 4. Preparar las entradas

- **Avatar:** EchoMimic no acepta transparencia, así que lo ponemos sobre el mismo fondo liso que la burbuja del montador.
- **Audio:** mientras no esté la voz real, voces de prueba con gTTS (las escenas que ya traen `audio_path` no se tocan).

In [ ]:
!pip install -q gTTS
from gtts import gTTS

os.makedirs("inputs", exist_ok=True)
img = Image.open(AVATAR_PNG).convert("RGBA")
base = Image.new("RGBA", img.size, (225, 233, 245, 255))
base.alpha_composite(img)
AVATAR_FLAT = os.path.abspath("inputs/avatar_flat.png")
base.convert("RGB").resize((512, 512), Image.LANCZOS).save(AVATAR_FLAT)

for s in spec["scenes"]:
    if not s.get("audio_path"):
        mp3 = f"inputs/{spec['company']}_{s['scene_id']:02d}.mp3"
        wav = mp3.replace(".mp3", ".wav")
        gTTS(s["text"], lang=spec.get("language", "es")).save(mp3)
        os.system(f"ffmpeg -y -loglevel error -i {mp3} -ar 16000 -ac 1 {wav}")
        s["audio_path"] = os.path.abspath(wav)
print([os.path.basename(s["audio_path"]) for s in spec["scenes"]])

## 5. Generar los vídeos con EchoMimic

Escribimos un archivo de configuración con nuestros casos (avatar + audio de cada escena) y lanzamos el script de inferencia acelerado dentro del entorno Python 3.10.

Empieza con `SOLO_PRUEBA = True` (la escena más corta) para comprobar que todo funciona.

In [ ]:
SOLO_PRUEBA = True

escenas = [s for s in spec["scenes"] if s.get("verified", True)]
if SOLO_PRUEBA:
    escenas = [min(escenas, key=lambda s: len(s["text"]))]

cfg = open("/content/EchoMimic/configs/prompts/animation_acc.yaml").read().split("## test cases")[0]
cfg += "## test cases\ntest_cases:\n"
cfg += f'  "{AVATAR_FLAT}":\n'
for s in escenas:
    cfg += f'    - "{s["audio_path"]}"\n'
open("/content/EchoMimic/configs/prompts/fintok.yaml", "w").write(cfg)
print(cfg.split("## test cases")[1])

In [ ]:
%cd /content/EchoMimic
!/content/emenv/bin/python -u infer_audio2vid_acc.py --config ./configs/prompts/fintok.yaml
%cd /content

## 6. Asociar cada vídeo a su escena

In [ ]:
os.makedirs("outputs/avatar_videos", exist_ok=True)
for s in spec["scenes"]:
    stem = os.path.splitext(os.path.basename(s["audio_path"]))[0]
    hits = sorted(glob.glob(f"/content/EchoMimic/output/**/*_{stem}_*withaudio.mp4", recursive=True))
    if hits:
        dst = f"outputs/avatar_videos/{spec['company']}_{spec['year']}_scene{s['scene_id']:02d}.mp4"
        shutil.copy(hits[-1], dst)
        s["avatar_video_path"] = dst
        print("✓ escena", s["scene_id"], "→", dst)
    else:
        print("· escena", s["scene_id"], "sin vídeo (usará el avatar de respaldo)")

with open("outputs/spec_with_avatar.json", "w", encoding="utf-8") as f:
    json.dump(spec, f, ensure_ascii=False, indent=2)

In [ ]:
from IPython.display import Video
primero = next(s["avatar_video_path"] for s in spec["scenes"] if s.get("avatar_video_path"))
Video(primero, embed=True, width=320)

## 7. Montar el vídeo final (opcional)

Sube `compositor.py` y los tres PNG del avatar para montarlo aquí mismo.

In [ ]:
uploaded = files.upload()
os.makedirs("assets/avatar", exist_ok=True)
for name in uploaded:
    if name.endswith(".png"):
        shutil.move(name, f"assets/avatar/{name}")

from compositor import render_video
out = render_video(spec, avatar_dir="assets/avatar", out_path=f"outputs/{spec['company']}_{spec['year']}_echomimic.mp4")
Video(out, embed=True, width=360)

## 8. Descargar resultados

In [ ]:
!zip -qr echomimic_resultados.zip outputs inputs
files.download("echomimic_resultados.zip")